In [20]:
import numpy as np

import torch
import torch.nn as nn

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [31]:
class ChessFormer(nn.Module):
    def __init__(self, vocab_size=13, embed_dim=256, n_heads=8, n_layers=4, dropout=0.1):
        super().__init__()
        
        # 1. Token Embeddings (13 piece types + padding/special if needed)
        self.piece_embedding = nn.Embedding(vocab_size, embed_dim)
        
        # 2. Positional Embeddings (Learnable, 64 squares)
        self.pos_embedding = nn.Parameter(torch.randn(1, 64, embed_dim))
        
        # 3. Class Token (CLS) to aggregate global board state
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        
        # 4. Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, 
            nhead=n_heads, 
            dim_feedforward=embed_dim * 4, 
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True # Pre-Norm is often more stable
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        
        # 5. Prediction Head (Regression)
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Tanh() # Force output to [-1, 1] to match our normalization
        )
        
    def forward(self, x):
        # x shape: (Batch, 64)
        B, T = x.shape
        
        # Embed Pieces
        x = self.piece_embedding(x) # (B, 64, Embed)
        
        # Add Positional Embedding
        x = x + self.pos_embedding
        
        # Append CLS token
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1) # (B, 65, Embed)
        
        # Transform
        x = self.encoder(x)
        
        # Extract CLS state (index 0) for regression
        cls_out = x[:, 0, :]
        
        # Predict
        val = self.head(cls_out)
        
        # Return both value (for loss) and latent (for your experiments)
        return val, cls_out

In [32]:
def load_model():
    model = ChessFormer(vocab_size=13, embed_dim=256, n_heads=8, n_layers=4, dropout=0.1)
    model.load_state_dict(torch.load("./models/chessformer_ep3.pth", weights_only=True, map_location=torch.device(device)))
    model.eval()
    return model

In [33]:
model = load_model()

/tmp/ipykernel_239841/3375567413.py:24: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


In [35]:
def parse_fen_batch(fens):
    """
    Parses a list of FEN strings into a numpy array (N, 64) of int8.
    This runs in parallel processes.
    """
    PIECE_TO_INT = {
        '.': 0, 'P': 1, 'N': 2, 'B': 3, 'R': 4, 'Q': 5, 'K': 6,
        'p': 7, 'n': 8, 'b': 9, 'r': 10, 'q': 11, 'k': 12
    }

    results = []
    for fen in fens:
        board_str = fen.split(" ")[0]
        # Expand empty squares '3' -> '...'
        for i in range(1, 9):
            board_str = board_str.replace(str(i), '.' * i)
        board_str = board_str.replace('/', '')
        
        # Map to integers
        # Note: FEN starts at Rank 8 (top), standard mapping usually aligns 0=a1.
        # Here we just need a consistent 64-len sequence.
        ranks = [PIECE_TO_INT[c] for c in board_str]
        results.append(ranks)
    return np.array(results, dtype=np.int8)

In [36]:
asd = parse_fen_batch(["2r4k/p4bp1/3npq2/1p1p4/5P2/P2B2R1/1P2Q2P/1K4R1 w - - 10 29",])
asd

array([[ 0,  0, 10,  0,  0,  0,  0, 12,  7,  0,  0,  0,  0,  9,  7,  0,
         0,  0,  0,  8,  7, 11,  0,  0,  0,  7,  0,  7,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  1,  0,  0,  1,  0,  0,  3,  0,  0,  4,  0,
         0,  1,  0,  0,  5,  0,  0,  1,  0,  6,  0,  0,  0,  0,  4,  0]],
      dtype=int8)

In [38]:
def get_status_eval(model, fens):
    """
    Given a list of FEN strings, returns their predicted evaluations.
    """
    # Parse FENs to (N, 64) int arrays
    board_arrays = parse_fen_batch(fens)
    
    # Convert to torch tensor
    board_tensors = torch.from_numpy(board_arrays).long().to(device)
    
    # Get model predictions
    with torch.no_grad():
        values, _ = model(board_tensors)
    
    return values.cpu().numpy().flatten()

In [40]:
get_status_eval(model, ["2r4k/p4bp1/3npq2/1p1p4/5P2/P2B2R1/1P2Q2P/1K4R1 w - - 10 29", 
                        "rnb1kbnr/pp3ppp/4p3/2pq4/3P4/5N2/PPPN1PPP/R1BQKB1R b KQkq - 1 5"])

array([0.6876356 , 0.11652429], dtype=float32)

In [ ]:
with torch.no_grad():
    # for b_in, target in val_loader:
    
    b_in, target = b_in.to(device), target.to(device)
    preds, _ = model(b_in)
    loss = loss_fn(preds, target)
    val_loss += loss.item()

In [41]:
import chess

In [72]:
def get_fen_4_posible_moves(fen):
    board = chess.Board(fen)
    legal_moves = list(board.legal_moves)
    legal_moves = [move for move in legal_moves]
    posible_fens = []
    for move in legal_moves:
        print(len(posible_fens))
        board = chess.Board(fen)
        board.push(move)
        posible_fens.append(board.fen())
    
    return posible_fens

In [74]:
get_status_eval(model, asd)

array([0.83858514, 0.68852973, 0.6737464 , 0.63843876, 0.6380331 ,
       0.64886045, 0.64573884, 0.6422421 , 0.6243134 , 0.6793628 ,
       0.67364204, 0.7417091 , 0.66137433, 0.68172896, 0.68582076,
       0.8302597 , 0.68865174, 0.72655994, 0.6653935 , 0.6934658 ,
       0.6817055 , 0.6642595 , 0.6864753 , 0.64185363, 0.64505553,
       0.6683666 , 0.6370969 , 0.63997036, 0.6457416 , 0.64936006,
       0.6242314 , 0.63519293, 0.62339437, 0.61456895, 0.6316756 ,
       0.6500472 , 0.73090255, 0.62342197, 0.6706481 , 0.63473177,
       0.6408346 , 0.6831478 , 0.6105558 ], dtype=float32)

In [66]:
import pandas as pd

In [73]:
asd = get_fen_4_posible_moves(fen)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42


In [ ]:
asd

In [42]:
board = chess.Board("2r4k/p4bp1/3npq2/1p1p4/5P2/P2B2R1/1P2Q2P/1K4R1 w - - 10 29")

In [56]:
fen = "2r4k/p4bp1/3npq2/1p1p4/5P2/P2B2R1/1P2Q2P/1K4R1 w - - 10 29"

In [46]:
list(board.legal_moves)[0].uci()

'g3g7'

In [60]:
fen = "2r4k/p4bp1/3npq2/1p1p4/5P2/P2B2R1/1P2Q2P/1K4R1 w - - 10 29"
board = chess.Board(fen)
legal_moves = list(board.legal_moves)
# legal_moves = [move.uci() for move in legal_moves]
board.push(legal_moves[0])

In [61]:
board.fen()

'2r4k/p4bR1/3npq2/1p1p4/5P2/P2B4/1P2Q2P/1K4R1 b - - 0 29'

In [53]:
legal_moves

[Move.from_uci('g3g7'),
 Move.from_uci('g3g6'),
 Move.from_uci('g3g5'),
 Move.from_uci('g3g4'),
 Move.from_uci('g3h3'),
 Move.from_uci('g3f3'),
 Move.from_uci('g3e3'),
 Move.from_uci('g3g2'),
 Move.from_uci('d3h7'),
 Move.from_uci('d3g6'),
 Move.from_uci('d3f5'),
 Move.from_uci('d3b5'),
 Move.from_uci('d3e4'),
 Move.from_uci('d3c4'),
 Move.from_uci('d3c2'),
 Move.from_uci('e2e6'),
 Move.from_uci('e2h5'),
 Move.from_uci('e2e5'),
 Move.from_uci('e2g4'),
 Move.from_uci('e2e4'),
 Move.from_uci('e2f3'),
 Move.from_uci('e2e3'),
 Move.from_uci('e2g2'),
 Move.from_uci('e2f2'),
 Move.from_uci('e2d2'),
 Move.from_uci('e2c2'),
 Move.from_uci('e2f1'),
 Move.from_uci('e2e1'),
 Move.from_uci('e2d1'),
 Move.from_uci('g1g2'),
 Move.from_uci('g1h1'),
 Move.from_uci('g1f1'),
 Move.from_uci('g1e1'),
 Move.from_uci('g1d1'),
 Move.from_uci('g1c1'),
 Move.from_uci('b1a2'),
 Move.from_uci('b1a1'),
 Move.from_uci('f4f5'),
 Move.from_uci('a3a4'),
 Move.from_uci('h2h3'),
 Move.from_uci('b2b3'),
 Move.from_uci('

In [ ]:

# minimax with alpha-beta pruning using the neural eval function

def minimax_ab(board, depth, alpha, beta, maximizing, model):
    """Return evaluation of the position from the current player's perspective.

    - ``board`` is a ``chess.Board`` instance that will be modified in place
      (push/pop moves).
    - ``depth`` is the number of plies remaining.
    - ``alpha`` / ``beta`` are the pruning bounds.
    - ``maximizing`` is True if we are searching for the side to move;
      False if we are searching for the opponent (i.e. we just made a move).
    - ``model`` is the neural network used by ``get_status_eval``.
    """
    # terminal condition
    if depth == 0 or board.is_game_over():
        fen = board.fen()
        return get_status_eval(model, [fen])[0]

    if maximizing:
        value = -float("inf")
        for move in board.legal_moves:
            board.push(move)
            score = minimax_ab(board, depth - 1, alpha, beta, False, model)
            board.pop()
            if score > value:
                value = score
            if score > alpha:
                alpha = score
            if beta <= alpha:
                break
        return value
    else:
        value = float("inf")
        for move in board.legal_moves:
            board.push(move)
            score = minimax_ab(board, depth - 1, alpha, beta, True, model)
            board.pop()
            if score < value:
                value = score
            if score < beta:
                beta = score
            if beta <= alpha:
                break
        return value


def find_best_move(board, depth, model):
    """Wrapper that returns the best move and its evaluation for ``board``.

    The evaluation is always from the side to move in ``board``.
    """
    best_move = None
    best_score = -float("inf")
    for move in board.legal_moves:
        board.push(move)
        score = minimax_ab(board, depth - 1, -float("inf"), float("inf"), True, model)
        board.pop()
        if score > best_score:
            best_score = score
            best_move = move
    return best_move, best_score


In [86]:
board = chess.Board("2r4k/p4bp1/3npq2/1p1p4/5P2/P2B2R1/1P2Q2P/1K4R1 w - - 10 29")
move, scoore = find_best_move(board, 3, model)
move, scoore

(Move.from_uci('g3h3'), np.float32(0.76195246))

In [80]:
move.uci()

'g3g7'

In [88]:
board = chess.Board("2r2k2/p4bpB/3npq2/1p1p4/5P2/P6R/1P2Q2P/1K4R1 w - - 14 31")
move, scoore = find_best_move(board, 3, model)
move, scoore

(Move.from_uci('e2e6'), np.float32(0.53594))

In [81]:

board = chess.Board("2r4k/p4bR1/4pq2/1p1p4/4nP2/P2B4/1P2Q2P/1K4R1 w - - 1 30")
move, scoore = find_best_move(board, 3, model)
move, scoore

(Move.from_uci('d3e4'), np.float32(0.8359828))

In [82]:

board = chess.Board("2r4k/p4bq1/4p3/1p1p4/4BP2/P7/1P2Q2P/1K4R1 w - - 0 31")
move, scoore = find_best_move(board, 3, model)
move, scoore

(Move.from_uci('g1g7'), np.float32(0.6440237))

In [84]:
board = chess.Board("2r5/p5k1/4p1b1/1p1B4/5P2/P7/1P2Q2P/1K6 w - - 1 33")
move, scoore = find_best_move(board, 3, model)
move, scoore

(Move.from_uci('e2d3'), np.float32(0.86899585))

In [ ]:
def get_fen_4_posible_moves(self, fen):
    board = chess.Board(fen)
    legal_moves = list(board.legal_moves)
    legal_moves = [move for move in legal_moves]
    posible_fens = []
    for move in legal_moves:
        board = chess.Board(fen)
        board.push(move)
        posible_fens.append(board.fen())
    
    return posible_fens